In [2]:
from importlib.metadata import version
print(f"pytorch version:", version("torch"))

pytorch version: 2.10.0+cu126


In [3]:
import os
import requests
from collections import Counter, deque
from functools import lru_cache
import re

In [4]:
if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as file:
        file.write(response.content)

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()


print(f"Total number of chracter : {len(raw_text)}")
print(raw_text[:100])

Total number of chracter : 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [6]:
text = "Hello world!, this is me."

result = re.split(r"(\s)", text)
print(result)

['Hello', ' ', 'world!,', ' ', 'this', ' ', 'is', ' ', 'me.']


In [7]:
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ' ', 'world!', ',', '', ' ', 'this', ' ', 'is', ' ', 'me', '.', '']


In [8]:
result = [item for item in result if item.strip()]
print(result)

['Hello', 'world!', ',', 'this', 'is', 'me', '.']


In [9]:
result = re.split(r'([!,.:;?_"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', 'world', '!', ',', 'this', 'is', 'me', '.']


In [10]:
preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [11]:
print(len(preprocessed))

4690


In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [13]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [14]:
class SimpleTokenizerv1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([.,:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [15]:
tokenizer = SimpleTokenizerv1(vocab)

text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [16]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [17]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [18]:
len(vocab.items())

1132

In [19]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [20]:
class SimpleTokenizerv2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\'])|--|\s', text)
        preprocessed = [item.strip() for item in preprocessed if item and item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r"\1", text)
        return text

In [21]:
tokenizer2 = SimpleTokenizerv2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of palace."
text = " <|endoftext|>".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|>In the sunlit terraces of palace.


In [22]:
tokenizer2.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1131, 988, 956, 984, 722, 1131, 7]

In [23]:
tokenizer2.decode(tokenizer2.encode(text))

'<|unk|>, do you like tea? <|unk|> the sunlit terraces of <|unk|>.'

Implementing BPE algorithm from scratch

First let's understand what are bytes

In [24]:
text = "This is some text"
byte_array = bytearray(text, "utf-8")
print(byte_array)

bytearray(b'This is some text')


When we call list() function on a bytes object each byte in the array is treated as an individual element and result is a list of integers corresponding to byte values.

In [25]:
ids = list(byte_array)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


This is a simple way to tokenize the text. But the problem with this approach is that every character has its corresponding byte and byte has its corresponding integer, so if a text is of 100 chracters then the lenght of tokens will be 100, so there would lot of hidden representations given as an input to the llm, due to which it won't capture long term dependencies.

In [26]:
print(f"Number of character : {len(text)}")
print(f"Number of tokens: {len(ids)}")

Number of character : 17
Number of tokens: 17


BPE tokenizers generally have a token ID for words or subwords not characters.

# Building the vocabulary

The goal of BPE tokenization algorithm is to build a vocabulary of commonly occuring subwords..

## BPE algorithm outline

### 1. Identify frequent pairs

 In each iteration, scan the text to find the most commonly occuring pair of bytes


### 2.Replace and Record

<ul>
<li>Replace that pair with new token id.</li>
<li>Save the mapping in the vocabulary.</li>
</ul>

### 3.Repeat until no further merging possible

<ul>
<li>Keep repeating steps 1 and 2 continously until no further words are left to merge.</li>
<li>Stop when no further compression is possible.</li>
</ul>

In [27]:
class BPETokenizer:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        self.bpe_merges = {}

    def train(self, text:str, vocab_size:int, allowed_specials:dict={"<|endoftext|>"}):
        processed_text = []
        # for char in text:
        #     processed_text.append(char)

        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(char for char in sorted(set(text)) if char not in unique_chars)

        self.vocab = {char:i for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {i:char for i, char in enumerate(unique_chars)}

        print(f"Allowed special")

        if allowed_specials:
            for token in allowed_specials:
                if token not in vocab:
                    new_id = len(self.vocab)
                    self.vocab[token] = new_id
                    self.inverse_vocab[new_id] = token
                
        token_ids = [self.vocab[char] for char in text]
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:
                print(f"Breaking before freq. pairs")
                break
            token_ids = self.replace_pair(token_ids, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

    def encode(self, text:str):
        tokens = []
        words = text.replace("\n", " \n ").split()
        for i, word in enumerate(words):
            if i>0 and not word.startswith("\n"):
                tokens.append(f" {word}")
            else:
                tokens.append(word)

        token_ids = []
        for token in tokens:
            if token in self.vocab:
                token_id = self.vocab[token]
                token_ids.append(token_id)
            else:
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids
    
    def tokenize_with_bpe(self, token:str):
        token_ids = [self.vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Chracters not found in vocab: {missing_chars}")
        
        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i+1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    i+=2
                    can_merge=True
                else:
                    new_tokens.append(token_ids[i])
                    i+= 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens
        
        return token_ids
    
    def decode(self, token_ids:list):
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.inverse_vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.inverse_vocab[token_id]
            decoded_string += token
        return decoded_string
    
    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)
    
    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        pairs = Counter(zip(token_ids, token_ids[1:]))

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalide mode. Choose 'most' or 'least'. ")
        
    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []
        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


In [28]:
tokenizer3 = BPETokenizer()
tokenizer3.train(raw_text, vocab_size=1000, allowed_specials={"<|endoftext|>"})

Allowed special
